# Prepare ML Training Data - Multi-Route

This notebook generalizes the single-route pipeline to handle any combination of cities.

## Configuration

**Input:**
- List of cities (must have corresponding weather feature files)
- Raw BITRE flight data

**Output:**
- `ml_training_data_multiroute.csv` - Combined flight + weather data for all route pairs

---

## Table of Contents

1. [Configuration](#1-configuration)
2. [Setup](#2-setup)
3. [Load and Clean Flight Data](#3-load-and-clean-flight-data)
4. [Load Weather Features](#4-load-weather-features)
5. [Merge Flight and Weather Data](#5-merge-flight-and-weather-data)
6. [Final Dataset Preparation](#6-final-dataset-preparation)
7. [Export](#7-export)
8. [Summary](#8-summary)

## 1. Configuration

Define the cities to include. Each city must have:
1. A weather features file: `data/processed/features_{city_code}.csv`
2. Flight routes in the BITRE data using the city's full name

In [1]:
# =============================================================================
# CONFIGURATION - Modify this section to add/remove cities
# =============================================================================

# City mapping: full name (as in BITRE data) -> code (as in weather files)
# Add new cities here as needed
CITY_MAPPING = {
    'Sydney': 'syd',
    'Melbourne': 'mel',
    'Hobart': 'hba',
    # Add more cities as weather data becomes available:
    # 'Brisbane': 'bne',
    # 'Perth': 'per',
    # 'Adelaide': 'adl',
}

# Select which cities to include in this run
# Must be a subset of CITY_MAPPING keys
CITIES_TO_INCLUDE = ['Sydney', 'Melbourne', 'Hobart']

# Output filename
OUTPUT_FILENAME = 'ml_training_data_multiroute.csv'

print(f"Cities to include: {CITIES_TO_INCLUDE}")
print(f"City codes: {[CITY_MAPPING[c] for c in CITIES_TO_INCLUDE]}")

Cities to include: ['Sydney', 'Melbourne', 'Hobart']
City codes: ['syd', 'mel', 'hba']


## 2. Setup

In [2]:
import pandas as pd
import numpy as np
import glob
import os
from itertools import permutations
from datetime import datetime
from dateutil.relativedelta import relativedelta
import requests

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

print("Setup complete")

Setup complete


In [3]:
# Generate all route pairs (permutations, not combinations - direction matters)
route_pairs = list(permutations(CITIES_TO_INCLUDE, 2))

print(f"Route pairs to process ({len(route_pairs)} routes):")
for dep, arr in route_pairs:
    print(f"  {dep} -> {arr}")

Route pairs to process (6 routes):
  Sydney -> Melbourne
  Sydney -> Hobart
  Melbourne -> Sydney
  Melbourne -> Hobart
  Hobart -> Sydney
  Hobart -> Melbourne


In [4]:
# Verify weather feature files exist for all cities
print("Checking weather feature files...")
missing_files = []

for city in CITIES_TO_INCLUDE:
    code = CITY_MAPPING[city]
    filepath = f"../data/processed/features_{code}.csv"
    if os.path.exists(filepath):
        print(f"  \u2713 {city} ({code}): {filepath}")
    else:
        print(f"  \u2717 {city} ({code}): FILE NOT FOUND - {filepath}")
        missing_files.append(filepath)

if missing_files:
    raise FileNotFoundError(f"Missing weather files: {missing_files}")
else:
    print("\nAll weather feature files found!")

Checking weather feature files...
  ✓ Sydney (syd): ../data/processed/features_syd.csv
  ✓ Melbourne (mel): ../data/processed/features_mel.csv
  ✓ Hobart (hba): ../data/processed/features_hba.csv

All weather feature files found!


## 3. Load and Clean Flight Data

Load raw BITRE data and filter to selected routes.

In [5]:
def download_bitre_data(save_path="../data/raw/"):
    """
    Download the latest BITRE flight data by trying recent months.
    """
    base_url = "https://www.bitre.gov.au/sites/default/files/documents/"
    current_date = datetime.now()
    
    headers = {
        'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36'
    }
    
    for months_back in range(1, 3):
        target_date = current_date - relativedelta(months=months_back)
        month_name = target_date.strftime("%B").lower()
        year = target_date.year
        
        filename = f"OTP_Time_Series_Master_Current_{month_name}_{year}.xlsx"
        url = base_url + filename
        
        print(f"Attempting to download: {filename}")
        
        try:
            response = requests.get(url, headers=headers, timeout=60)
            if response.status_code == 200:
                filepath = save_path + filename
                with open(filepath, 'wb') as f:
                    f.write(response.content)
                print(f"\u2713 Successfully downloaded: {filename}")
                return filepath
            else:
                print(f"  Not found (HTTP {response.status_code})")
        except requests.RequestException as e:
            print(f"  Failed: {type(e).__name__}")
    
    print("\n\u26a0 Automatic download failed. Will use existing local file.")
    return None

# Attempt to download latest data
downloaded_file = download_bitre_data()
print(f"\n{'='*80}")

Attempting to download: OTP_Time_Series_Master_Current_december_2025.xlsx
  Not found (HTTP 404)
Attempting to download: OTP_Time_Series_Master_Current_november_2025.xlsx
✓ Successfully downloaded: OTP_Time_Series_Master_Current_november_2025.xlsx



In [6]:
# Load BITRE data
if downloaded_file:
    data_file = downloaded_file
else:
    local_files = glob.glob("../data/raw/OTP_Time_Series_Master_Current_*.xlsx")
    if local_files:
        data_file = max(local_files)
        print(f"Using existing local file: {data_file}")
    else:
        raise FileNotFoundError("No BITRE data file found.")

# Load all sheets
all_sheets_temp = pd.read_excel(data_file, sheet_name=None)
df_raw = pd.concat(all_sheets_temp.values(), ignore_index=True)

# Remove duplicates
rows_before = len(df_raw)
df_raw = df_raw.drop_duplicates()
rows_dropped = rows_before - len(df_raw)
if rows_dropped > 0:
    print(f"Removed {rows_dropped} duplicate rows")

# Convert date
df_raw['Month'] = pd.to_datetime(df_raw['Month'], format='%m/%Y')
df_raw['year_month'] = df_raw['Month'].dt.to_period('M').astype(str)

print(f"Loaded data from: {data_file}")
print(f"Total records: {len(df_raw)}")
print(f"Date range: {df_raw['year_month'].min()} to {df_raw['year_month'].max()}")

Removed 4198 duplicate rows
Loaded data from: ../data/raw/OTP_Time_Series_Master_Current_november_2025.xlsx
Total records: 90968
Date range: 2010-01 to NaT


In [7]:
# Build route filter: all permutations of selected cities
route_names = []
for dep, arr in route_pairs:
    route_names.append(f"{dep}-{arr}")

print(f"Filtering for routes: {route_names}")

# Filter to selected routes and exclude "All Airlines" aggregate
df_flights = df_raw[
    (df_raw['Route'].isin(route_names)) &
    (df_raw['Airline'] != 'All Airlines')
].copy()

print(f"\nFiltered to {len(df_flights)} records")
print(f"\nRoutes found:")
print(df_flights['Route'].value_counts())

Filtering for routes: ['Sydney-Melbourne', 'Sydney-Hobart', 'Melbourne-Sydney', 'Melbourne-Hobart', 'Hobart-Sydney', 'Hobart-Melbourne']

Filtered to 4252 records

Routes found:
Route
Melbourne-Sydney    752
Sydney-Melbourne    752
Hobart-Melbourne    741
Melbourne-Hobart    741
Hobart-Sydney       633
Sydney-Hobart       633
Name: count, dtype: int64


In [8]:
# Data cleaning steps
print("Cleaning flight data...")
print(f"\n{'='*80}")

# 1. Remove rows with Sectors Flown = 0
rows_before = len(df_flights)
df_flights = df_flights[df_flights['Sectors Flown'] > 0].copy()
print(f"Removed {rows_before - len(df_flights)} rows with Sectors Flown = 0")

# 2. Calculate target variables
df_flights['delay_rate'] = (df_flights['Sectors Flown'] - df_flights['Arrivals On Time']) / df_flights['Sectors Flown']
df_flights['is_high_delay'] = (df_flights['delay_rate'] > 0.25).astype(int)

# 3. Extract year
df_flights['year'] = df_flights['Month'].dt.year

# 4. Exclude COVID period (April-December 2020)
covid_mask = (df_flights['Month'] >= '2020-04-01') & (df_flights['Month'] <= '2020-12-31')
rows_before = len(df_flights)
df_flights = df_flights[~covid_mask].copy()
print(f"Removed {rows_before - len(df_flights)} rows from COVID period (Apr-Dec 2020)")

# 5. Rename columns
column_mapping = {
    'Route': 'route',
    'Departing Port': 'departing_port',
    'Arriving Port': 'arriving_port',
    'Airline': 'airline',
    'Month': 'month',
    'Sectors Scheduled': 'sectors_scheduled',
    'Sectors Flown': 'sectors_flown',
    'Cancellations': 'cancellations',
    'Arrivals On Time': 'arrivals_on_time',
    'Arrivals Delayed': 'arrivals_delayed',
    'Cancellations \n\n(%)': 'cancellations_pct',
}
df_flights = df_flights.rename(columns=column_mapping)

print(f"\nFinal flight data: {len(df_flights)} records")
print(f"Date range: {df_flights['year_month'].min()} to {df_flights['year_month'].max()}")

Cleaning flight data...

Removed 10 rows with Sectors Flown = 0
Removed 102 rows from COVID period (Apr-Dec 2020)

Final flight data: 4140 records
Date range: 2010-01 to 2025-11


In [9]:
# Summary by route
print("Records per route:")
print(df_flights.groupby(['departing_port', 'arriving_port']).size().reset_index(name='count'))
print(f"\n{'='*80}")

print("\nAirlines per route:")
for (dep, arr), group in df_flights.groupby(['departing_port', 'arriving_port']):
    airlines = group['airline'].unique()
    print(f"  {dep} -> {arr}: {len(airlines)} airlines")

Records per route:
  departing_port arriving_port  count
0         Hobart     Melbourne    730
1         Hobart        Sydney    617
2      Melbourne        Hobart    730
3      Melbourne        Sydney    723
4         Sydney        Hobart    617
5         Sydney     Melbourne    723


Airlines per route:
  Hobart -> Melbourne: 6 airlines
  Hobart -> Sydney: 4 airlines
  Melbourne -> Hobart: 6 airlines
  Melbourne -> Sydney: 7 airlines
  Sydney -> Hobart: 4 airlines
  Sydney -> Melbourne: 7 airlines


## 4. Load Weather Features

Load weather feature files for all cities.

In [10]:
# Load weather features for each city
weather_data = {}

for city in CITIES_TO_INCLUDE:
    code = CITY_MAPPING[city]
    filepath = f"../data/processed/features_{code}.csv"
    
    df_weather = pd.read_csv(filepath)
    weather_data[city] = df_weather
    
    print(f"{city} ({code}): {len(df_weather)} months, {len(df_weather.columns)-1} features")
    print(f"  Date range: {df_weather['year_month'].min()} to {df_weather['year_month'].max()}")

print(f"\n{'='*80}")
print(f"Loaded weather data for {len(weather_data)} cities")

Sydney (syd): 204 months, 21 features
  Date range: 2009-01 to 2025-12
Melbourne (mel): 205 months, 21 features
  Date range: 2009-01 to 2026-01
Hobart (hba): 205 months, 21 features
  Date range: 2009-01 to 2026-01

Loaded weather data for 3 cities


In [11]:
# Get weather feature column names (same across all cities)
sample_city = CITIES_TO_INCLUDE[0]
weather_feature_cols = [col for col in weather_data[sample_city].columns if col != 'year_month']

print(f"Weather feature columns ({len(weather_feature_cols)}):")
for col in weather_feature_cols:
    print(f"  - {col}")

Weather feature columns (21):
  - days_in_month
  - max_temperature
  - avg_max_temp
  - min_temperature
  - avg_min_temp
  - max_daily_rainfall
  - avg_rainfall_per_day
  - avg_wind_speed
  - max_wind_speed
  - avg_max_humidity
  - avg_min_humidity
  - temp_range_mean
  - days_above_35C
  - temp_volatility
  - rainy_days
  - heavy_rain_days
  - avg_rainfall_on_rainy_days
  - days_high_wind
  - wind_speed_std
  - days_high_humidity
  - extreme_weather_days


## 5. Merge Flight and Weather Data

For each flight record, merge weather data from both departure and arrival airports.

In [12]:
def prepare_weather_with_suffix(df_weather, suffix):
    """
    Prepare weather dataframe with column suffix (_dep or _arr).
    """
    df = df_weather.copy()
    rename_dict = {col: f"{col}{suffix}" for col in df.columns if col != 'year_month'}
    df = df.rename(columns=rename_dict)
    return df

# Prepare departure and arrival weather dataframes for each city
weather_dep = {city: prepare_weather_with_suffix(weather_data[city], '_dep') for city in CITIES_TO_INCLUDE}
weather_arr = {city: prepare_weather_with_suffix(weather_data[city], '_arr') for city in CITIES_TO_INCLUDE}

print("Weather dataframes prepared with _dep and _arr suffixes")
print(f"Example columns for {CITIES_TO_INCLUDE[0]} departure: {weather_dep[CITIES_TO_INCLUDE[0]].columns[:5].tolist()}...")

Weather dataframes prepared with _dep and _arr suffixes
Example columns for Sydney departure: ['year_month', 'days_in_month_dep', 'max_temperature_dep', 'avg_max_temp_dep', 'min_temperature_dep']...


In [13]:
# Merge weather data based on departing and arriving ports
print("Merging weather data...")
print(f"\n{'='*80}")

merged_dfs = []

for (dep_city, arr_city) in route_pairs:
    # Filter flights for this route
    route_mask = (df_flights['departing_port'] == dep_city) & (df_flights['arriving_port'] == arr_city)
    df_route = df_flights[route_mask].copy()
    
    if len(df_route) == 0:
        print(f"  {dep_city} -> {arr_city}: No flights found, skipping")
        continue
    
    # Merge departure weather
    df_route = df_route.merge(weather_dep[dep_city], on='year_month', how='left')
    
    # Merge arrival weather
    df_route = df_route.merge(weather_arr[arr_city], on='year_month', how='left')
    
    merged_dfs.append(df_route)
    
    # Count nulls
    null_dep = df_route[[c for c in df_route.columns if c.endswith('_dep')]].isnull().sum().sum()
    null_arr = df_route[[c for c in df_route.columns if c.endswith('_arr')]].isnull().sum().sum()
    
    print(f"  {dep_city} -> {arr_city}: {len(df_route)} rows (null dep: {null_dep}, null arr: {null_arr})")

# Combine all routes
df_combined = pd.concat(merged_dfs, ignore_index=True)
print(f"\n{'='*80}")
print(f"Combined dataset: {len(df_combined)} rows")

Merging weather data...

  Sydney -> Melbourne: 723 rows (null dep: 0, null arr: 0)
  Sydney -> Hobart: 617 rows (null dep: 0, null arr: 0)
  Melbourne -> Sydney: 723 rows (null dep: 0, null arr: 0)
  Melbourne -> Hobart: 730 rows (null dep: 0, null arr: 0)
  Hobart -> Sydney: 617 rows (null dep: 0, null arr: 0)
  Hobart -> Melbourne: 730 rows (null dep: 0, null arr: 0)

Combined dataset: 4140 rows


In [14]:
# Check for null values
null_counts = df_combined.isnull().sum()
cols_with_nulls = null_counts[null_counts > 0]

if len(cols_with_nulls) > 0:
    print("Columns with null values after merge:")
    print(cols_with_nulls)
    print(f"\n{'='*80}")
    
    # Show which year_months have missing data
    null_rows = df_combined[df_combined.isnull().any(axis=1)]
    print(f"Rows with null values: {len(null_rows)}")
    print(f"Year months with missing data: {sorted(null_rows['year_month'].unique())[:10]}...")
else:
    print("No null values found after merge")

No null values found after merge


## 6. Final Dataset Preparation

In [15]:
# Define column order
flight_cols = ['departing_port', 'arriving_port', 'airline', 'month', 'year_month', 'year',
               'sectors_scheduled', 'sectors_flown', 'arrivals_on_time', 'arrivals_delayed',
               'cancellations', 'cancellations_pct']

target_cols = ['delay_rate', 'is_high_delay']

weather_dep_cols = sorted([c for c in df_combined.columns if c.endswith('_dep')])
weather_arr_cols = sorted([c for c in df_combined.columns if c.endswith('_arr')])

# Select only columns that exist
flight_cols = [c for c in flight_cols if c in df_combined.columns]

print(f"Flight metadata columns: {len(flight_cols)}")
print(f"Target columns: {len(target_cols)}")
print(f"Departure weather features: {len(weather_dep_cols)}")
print(f"Arrival weather features: {len(weather_arr_cols)}")

# Reorder columns
column_order = flight_cols + target_cols + weather_dep_cols + weather_arr_cols
df_final = df_combined[column_order].copy()

# Sort by date, airline, and route
df_final = df_final.sort_values(['year_month', 'airline', 'departing_port', 'arriving_port']).reset_index(drop=True)

print(f"\nFinal dataset shape: {df_final.shape}")

Flight metadata columns: 12
Target columns: 2
Departure weather features: 21
Arrival weather features: 21

Final dataset shape: (4140, 56)


In [16]:
# Summary statistics
print("Dataset Summary")
print(f"{'='*80}")
print(f"\nTotal records: {len(df_final)}")
print(f"Date range: {df_final['year_month'].min()} to {df_final['year_month'].max()}")
print(f"\nRoutes:")
print(df_final.groupby(['departing_port', 'arriving_port']).size().reset_index(name='count'))

print(f"\nAirlines: {df_final['airline'].nunique()}")
print(df_final['airline'].value_counts())

print(f"\n{'='*80}")
print("\nTarget variable summary:")
print(f"\ndelay_rate:")
print(df_final['delay_rate'].describe())
print(f"\nis_high_delay distribution:")
print(df_final['is_high_delay'].value_counts())

Dataset Summary

Total records: 4140
Date range: 2010-01 to 2025-11

Routes:
  departing_port arriving_port  count
0         Hobart     Melbourne    730
1         Hobart        Sydney    617
2      Melbourne        Hobart    730
3      Melbourne        Sydney    723
4         Sydney        Hobart    617
5         Sydney     Melbourne    723

Airlines: 7
airline
Jetstar               1092
Virgin Australia      1092
Qantas                 873
QantasLink             531
Tigerair Australia     454
Rex Airlines            90
Regional Express         8
Name: count, dtype: int64


Target variable summary:

delay_rate:
count    4140.000000
mean        0.241056
std         0.132219
min         0.000000
25%         0.150685
50%         0.226394
75%         0.318872
max         1.000000
Name: delay_rate, dtype: float64

is_high_delay distribution:
is_high_delay
0    2405
1    1735
Name: count, dtype: int64


In [17]:
# Preview final dataset
print("Preview of final dataset:")
df_final.head(10)

Preview of final dataset:


,departing_port,arriving_port,airline,month,year_month,year,sectors_scheduled,sectors_flown,arrivals_on_time,arrivals_delayed,cancellations,cancellations_pct,delay_rate,is_high_delay,avg_max_humidity_dep,avg_max_temp_dep,avg_min_humidity_dep,avg_min_temp_dep,avg_rainfall_on_rainy_days_dep,avg_rainfall_per_day_dep,avg_wind_speed_dep,days_above_35C_dep,days_high_humidity_dep,days_high_wind_dep,days_in_month_dep,extreme_weather_days_dep,heavy_rain_days_dep,max_daily_rainfall_dep,max_temperature_dep,max_wind_speed_dep,min_temperature_dep,rainy_days_dep,temp_range_mean_dep,temp_volatility_dep,wind_speed_std_dep,avg_max_humidity_arr,avg_max_temp_arr,avg_min_humidity_arr,avg_min_temp_arr,avg_rainfall_on_rainy_days_arr,avg_rainfall_per_day_arr,avg_wind_speed_arr,days_above_35C_arr,days_high_humidity_arr,days_high_wind_arr,days_in_month_arr,extreme_weather_days_arr,heavy_rain_days_arr,max_daily_rainfall_arr,max_temperature_arr,max_wind_speed_arr,min_temperature_arr,rainy_days_arr,temp_range_mean_arr,temp_volatility_arr,wind_speed_std_arr
0,Hobart,Melbourne,Jetstar,2010-01-01,2010-01,2010,131.0,131.0,104.0,27.0,0.0,0.000000,0.206107,0,80.870968,23.803226,34.612903,12.264516,1.52,0.245161,5.320968,2.0,0,0.0,31,2,0.0,5.2,36.0,7.53,7.2,5.0,11.538710,4.280000,0.985090,86.451613,27.003226,32.322581,13.922581,5.36,0.864516,5.651290,6.0,7,3.0,31,12,1.0,14.0,42.5,10.14,8.6,5.0,13.080645,5.476667,1.690102
1,Hobart,Sydney,Jetstar,2010-01-01,2010-01,2010,59.0,59.0,51.0,8.0,0.0,0.000000,0.135593,0,80.870968,23.803226,34.612903,12.264516,1.52,0.245161,5.320968,2.0,0,0.0,31,2,0.0,5.2,36.0,7.53,7.2,5.0,11.538710,4.280000,0.985090,89.709677,27.958065,52.419355,20.587097,3.30,0.851613,6.013226,1.0,18,2.0,31,7,1.0,11.6,42.5,9.50,15.6,8.0,7.370968,3.963333,1.468147
2,Melbourne,Hobart,Jetstar,2010-01-01,2010-01,2010,131.0,131.0,105.0,26.0,0.0,0.000000,0.198473,0,86.451613,27.003226,32.322581,13.922581,5.36,0.864516,5.651290,6.0,7,3.0,31,12,1.0,14.0,42.5,10.14,8.6,5.0,13.080645,5.476667,1.690102,80.870968,23.803226,34.612903,12.264516,1.52,0.245161,5.320968,2.0,0,0.0,31,2,0.0,5.2,36.0,7.53,7.2,5.0,11.538710,4.280000,0.985090
3,Melbourne,Sydney,Jetstar,2010-01-01,2010-01,2010,150.0,147.0,110.0,37.0,3.0,2.000000,0.251701,1,86.451613,27.003226,32.322581,13.922581,5.36,0.864516,5.651290,6.0,7,3.0,31,12,1.0,14.0,42.5,10.14,8.6,5.0,13.080645,5.476667,1.690102,89.709677,27.958065,52.419355,20.587097,3.30,0.851613,6.013226,1.0,18,2.0,31,7,1.0,11.6,42.5,9.50,15.6,8.0,7.370968,3.963333,1.468147
4,Sydney,Hobart,Jetstar,2010-01-01,2010-01,2010,59.0,59.0,53.0,6.0,0.0,0.000000,0.101695,0,89.709677,27.958065,52.419355,20.587097,3.30,0.851613,6.013226,1.0,18,2.0,31,7,1.0,11.6,42.5,9.50,15.6,8.0,7.370968,3.963333,1.468147,80.870968,23.803226,34.612903,12.264516,1.52,0.245161,5.320968,2.0,0,0.0,31,2,0.0,5.2,36.0,7.53,7.2,5.0,11.538710,4.280000,0.985090
5,Sydney,Melbourne,Jetstar,2010-01-01,2010-01,2010,146.0,144.0,122.0,22.0,2.0,1.369863,0.152778,0,89.709677,27.958065,52.419355,20.587097,3.30,0.851613,6.013226,1.0,18,2.0,31,7,1.0,11.6,42.5,9.50,15.6,8.0,7.370968,3.963333,1.468147,86.451613,27.003226,32.322581,13.922581,5.36,0.864516,5.651290,6.0,7,3.0,31,12,1.0,14.0,42.5,10.14,8.6,5.0,13.080645,5.476667,1.690102
6,Hobart,Melbourne,Qantas,2010-01-01,2010-01,2010,65.0,65.0,58.0,7.0,0.0,0.000000,0.107692,0,80.870968,23.803226,34.612903,12.264516,1.52,0.245161,5.320968,2.0,0,0.0,31,2,0.0,5.2,36.0,7.53,7.2,5.0,11.538710,4.280000,0.985090,86.451613,27.003226,32.322581,13.922581,5.36,0.864516,5.651290,6.0,7,3.0,31,12,1.0,14.0,42.5,10.14,8.6,5.0,13.080645,5.476667,1.690102
7,Hobart,Sydney,Qantas,2010-01-01,2010-01,2010,35.0,35.0,30.0,5.0,0.0,0.000000,0.142857,0,80.870968,23.803226,34.612903,12.264516,1.52,0.245161,5.320968,2.0,0,0.0,31,2,0.0,5.2,36.0,7.53,7.2,5.0,11.538710,4.280000,0.985090,89.709677,27.958065,52.419355,20.587097,3.30,0.851613,6.013226,1.0,18,2.0,31,7,1.0,11.6,42.5,9.50,15.6,8.0,7.370968,3.963333,1.468147
8,Melbourne,Hobart,Qantas,2010-01-01,2010-01,2010,65.0,65.0,5

## 7. Export

In [18]:
# Save to CSV
output_path = f"../data/processed/{OUTPUT_FILENAME}"
df_final.to_csv(output_path, index=False)

print(f"ML training data saved to: {output_path}")
print(f"File size: {os.path.getsize(output_path) / 1024:.1f} KB")

ML training data saved to: ../data/processed/ml_training_data_multiroute.csv
File size: 2159.0 KB


## 8. Summary

**Data Combination Steps Completed:**

1. Configured cities and generated all route pair permutations
2. Loaded and cleaned raw BITRE flight data
3. Filtered to selected routes, excluded COVID period
4. Loaded weather features for all cities
5. Merged weather data based on flight direction:
   - Departure airport weather (suffix: `_dep`)
   - Arrival airport weather (suffix: `_arr`)
6. Combined all routes into single dataset
7. Saved final ML training data

**To add more cities:**
1. Ensure weather feature file exists: `data/processed/features_{city_code}.csv`
2. Add city to `CITY_MAPPING` dictionary
3. Add city to `CITIES_TO_INCLUDE` list
4. Re-run notebook